# Préparation des données `Lieux` — BAAC 2020–2024

Ce notebook est consacré à la préparation des données relatives aux lieux des accidents corporels de la circulation pour la période 2020–2024.

L'objectif est de vérifier la compatibilité des fichiers annuels, de les consolider, d'évaluer leur qualité, de sélectionner les variables pertinentes pour la problématique, puis d'appliquer les traitements nécessaires afin d'obtenir un jeu de données nettoyé et exploitable pour les prochaines étapes du projet.

## 1. Préparation et consolidation des données
### 1.1. Chargement des fichiers annuels

Les fichiers `Lieux` des années 2020 à 2024 sont chargés séparément afin de vérifier leur structure et leur compatibilité avant leur consolidation.

In [1257]:
# 1.1. Chargement des fichiers annuels

import pandas as pd

df_lieux_2020 = pd.read_csv("data/raw/2020/lieux-2020.csv", sep=";")
df_lieux_2021 = pd.read_csv("data/raw/2021/lieux-2021.csv", sep=";")
df_lieux_2022 = pd.read_csv("data/raw/2022/lieux-2022.csv", sep=";")
df_lieux_2023 = pd.read_csv("data/raw/2023/lieux-2023.csv", sep=";")
df_lieux_2024 = pd.read_csv("data/raw/2024/lieux-2024.csv", sep=";")

C:\Users\nabil\AppData\Local\Temp\ipykernel_19296\2962806035.py:7: DtypeWarning: Columns (0: nbv) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lieux_2022 = pd.read_csv("data/raw/2022/lieux-2022.csv", sep=";")
C:\Users\nabil\AppData\Local\Temp\ipykernel_19296\2962806035.py:8: DtypeWarning: Columns (0: lartpc) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lieux_2023 = pd.read_csv("data/raw/2023/lieux-2023.csv", sep=";")


### 1.2. Vérification de la structure des fichiers annuels

La structure des fichiers `Lieux` est comparée afin de vérifier leur compatibilité avant leur consolidation.

In [1258]:
# 1.2. Vérification de la structure des fichiers annuels

dfs_lieux = {
    2020: df_lieux_2020,
    2021: df_lieux_2021,
    2022: df_lieux_2022,
    2023: df_lieux_2023,
    2024: df_lieux_2024
}

for annee, df in dfs_lieux.items():
    print(f"{annee} : {df.shape}")
    print(df.columns.tolist())
    print()

2020 : (47744, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']

2021 : (56518, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']

2022 : (55302, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']

2023 : (70860, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']

2024 : (70248, 18)
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']



#### Résultat

Les cinq fichiers `Lieux` présentent une structure homogène : chacun contient **18 variables avec les mêmes noms de colonnes et dans le même ordre**.

Aucune harmonisation de nom de variable n'est nécessaire avant la consolidation des données 2020–2024.

### 1.3. Vérification de l'identifiant `Num_Acc`

La variable `Num_Acc` permet de relier le fichier `Lieux` aux autres fichiers BAAC.

Le nombre d'identifiants uniques est comparé au nombre de lignes afin de vérifier la granularité des fichiers annuels et d'identifier la présence éventuelle de plusieurs enregistrements pour un même accident.

In [1259]:
# 1.3. Vérification de l'identifiant Num_Acc

for annee, df in dfs_lieux.items():
    print(
        f"{annee} : "
        f"{len(df)} lignes | "
        f"{df['Num_Acc'].nunique()} identifiants uniques | "
        f"{df['Num_Acc'].isna().sum()} manquants"
    )

2020 : 47744 lignes | 47744 identifiants uniques | 0 manquants
2021 : 56518 lignes | 56518 identifiants uniques | 0 manquants
2022 : 55302 lignes | 55302 identifiants uniques | 0 manquants
2023 : 70860 lignes | 54822 identifiants uniques | 0 manquants
2024 : 70248 lignes | 54402 identifiants uniques | 0 manquants


#### Résultat

La variable `Num_Acc` ne contient aucune valeur manquante.

De 2020 à 2022, chaque accident correspond à une seule ligne dans le fichier `Lieux`.

En 2023 et 2024, certains accidents apparaissent sur plusieurs lignes. L'analyse de ces répétitions a montré qu'elles concernent les accidents en intersection et correspondent à la description de plusieurs voies associées à un même accident.

Ces répétitions ne sont donc pas considérées automatiquement comme des doublons. Leur traitement sera réévalué après la sélection des variables pertinentes pour l'analyse.

### 1.4. Vérification de la compatibilité des types

Les types de données sont comparés entre les fichiers annuels afin d'identifier les éventuelles différences de représentation avant leur consolidation.

In [1260]:
# 1.4. Vérification de la compatibilité des types

for annee, dataframe in dfs_lieux.items():
    print(f"{annee}")
    print(dataframe.dtypes)
    print()

2020
Num_Acc    int64
catr       int64
voie         str
v1         int64
v2           str
circ       int64
nbv        int64
vosp       int64
prof       int64
pr           str
pr1          str
plan       int64
lartpc       str
larrout      str
surf       int64
infra      int64
situ       int64
vma        int64
dtype: object

2021
Num_Acc    int64
catr       int64
voie         str
v1         int64
v2           str
circ       int64
nbv        int64
vosp       int64
prof       int64
pr           str
pr1          str
plan       int64
lartpc       str
larrout      str
surf       int64
infra      int64
situ       int64
vma        int64
dtype: object

2022
Num_Acc     int64
catr        int64
voie          str
v1          int64
v2            str
circ        int64
nbv        object
vosp        int64
prof        int64
pr            str
pr1           str
plan        int64
lartpc        str
larrout       str
surf        int64
infra       int64
situ        int64
vma         int64
dtype: object

2023

#### Résultat

Les types de données sont globalement compatibles entre les cinq fichiers annuels.

Certaines différences de représentation sont néanmoins observées selon les années, notamment pour `nbv` et `lartpc`.

La variable `nbv`, retenue pour l'analyse, fera l'objet d'un contrôle lors du nettoyage. La variable `lartpc` sera écartée lors de la sélection des variables et ne nécessite donc pas de traitement approfondi.

### 1.5. Consolidation des fichiers 2020–2024

Les cinq fichiers annuels sont regroupés dans un seul DataFrame afin de disposer d'un jeu de données `Lieux` couvrant l'ensemble de la période 2020–2024.

In [1261]:
# 1.5. Consolidation des fichiers 2020–2024

df_lieux = pd.concat(
    dfs_lieux.values(),
    ignore_index=True
)

df_lieux.shape

(300672, 18)

#### Résultat

La consolidation des cinq fichiers annuels a permis d'obtenir un DataFrame contenant **300 672 lignes et 18 variables**.

Le nombre de lignes correspond à la somme des observations des fichiers 2020 à 2024. La consolidation s'est donc effectuée correctement.

### 1.6. Sauvegarde des données consolidées

Le jeu de données consolidé est enregistré dans le dossier `interim` avant les opérations de nettoyage.

In [1262]:
# 1.6. Sauvegarde des données consolidées

df_lieux.to_csv(
    "data/interim/lieux_2020_2024.csv",
    sep=";",
    index=False
)


## 2. Évaluation de la qualité et sélection des variables

Cette étape vise à identifier les éventuels problèmes de qualité du jeu de données consolidé, puis à sélectionner les variables pertinentes pour l'analyse des facteurs associés à la gravité des accidents.

Les contrôles détaillés seront ensuite concentrés sur les variables retenues afin d'éviter des traitements inutiles.

### 2.1. Contrôle des doublons complets

Les doublons complets correspondent à des lignes strictement identiques sur l'ensemble des variables du fichier `Lieux`.

In [1263]:
# 2.1. Contrôle des doublons

df_lieux.duplicated().sum()

np.int64(2)

In [1264]:
# Affichage des doublons complets

df_lieux[
    df_lieux.duplicated(keep=False)
].sort_values("Num_Acc")

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
246308,202400012279,3,NaN,-1,NaN,2,-1,-1,2,-1,-1,1,NaN,-1,1,0,1,-1
246309,202400012279,3,NaN,-1,NaN,2,-1,-1,2,-1,-1,1,NaN,-1,1,0,1,-1
287766,202400044389,4,NaN,-1,NaN,2,-1,-1,1,-1,-1,1,NaN,-1,1,0,1,-1
287767,202400044389,4,NaN,-1,NaN,2,-1,-1,1,-1,-1,1,NaN,-1,1,0,1,-1


#### Résultat

Le jeu de données contient **2 doublons complets**.

Ils concernent les accidents `202400012279` et `202400044389`, pour lesquels deux lignes présentent exactement les mêmes valeurs sur l'ensemble des variables.

Ces doublons complets seront supprimés lors de l'étape de nettoyage.

### 2.2. Sélection métier des variables

Les variables du fichier `Lieux` sont sélectionnées en fonction de leur intérêt pour l'analyse des facteurs associés à la gravité des accidents.

Les variables principalement destinées à l'identification précise d'une voie, à son référencement technique ou jugées peu pertinentes pour la problématique ne sont pas retenues.

Cette sélection permet de concentrer les contrôles qualité et les traitements sur les variables réellement utiles à l'analyse.

| Variable | Information apportée | Décision |
|---|---|---|
| `Num_Acc` | Identifiant de l'accident et liaison entre les tables | Garder |
| `catr` | Catégorie de route | Garder |
| `voie` | Identification précise de la voie | Supprimer |
| `v1` | Indice numérique de la voie | Supprimer |
| `v2` | Indice alphanumérique de la voie | Supprimer |
| `circ` | Régime de circulation | Garder |
| `nbv` | Nombre de voies de circulation | Garder |
| `vosp` | Voie réservée | Supprimer |
| `prof` | Profil de la route | Garder |
| `pr` | Point de repère routier | Supprimer |
| `pr1` | Distance au point de repère | Supprimer |
| `plan` | Tracé de la route | Garder |
| `lartpc` | Largeur du terre-plein central | Supprimer |
| `larrout` | Largeur de la chaussée | Supprimer |
| `surf` | État de la surface de la chaussée | Garder |
| `infra` | Infrastructure ou aménagement particulier | Supprimer |
| `situ` | Situation de l'accident sur la chaussée | Garder |
| `vma` | Vitesse maximale autorisée | Garder |

#### Décision

Les **9 variables retenues** sont :

`Num_Acc`, `catr`, `circ`, `nbv`, `prof`, `plan`, `surf`, `situ` et `vma`.

La variable `larrout` n'est finalement pas retenue en raison de son taux très élevé de valeurs non renseignées, qui limite fortement son exploitation pour l'analyse plus de 80%.

Les contrôles qualité détaillés et les traitements de nettoyage seront désormais concentrés sur les variables retenues.

### 2.3. Simulation de la sélection des variables

La sélection des variables est simulée sans modifier le DataFrame consolidé afin d'évaluer son impact sur les répétitions de lignes.

Certaines lignes correspondant à plusieurs voies peuvent devenir identiques lorsque les variables d'identification non retenues sont supprimées.

In [1265]:
# 2.3. Simulation de la sélection des variables

colonnes_lieux_retenues = [
    "Num_Acc",
    "catr",
    "circ",
    "nbv",
    "prof",
    "plan",
    "surf",
    "situ",
    "vma"
]

df_lieux_selection = df_lieux[colonnes_lieux_retenues].copy()

print("Nombre de lignes :", len(df_lieux_selection))
print(
    "Doublons complets après sélection :",
    df_lieux_selection.duplicated().sum()
)

Nombre de lignes : 300672
Doublons complets après sélection : 6234


In [1266]:
# Vérification des Num_Acc restant multiples après suppression simulée des doublons

df_lieux_selection_sans_doublons = (
    df_lieux_selection.drop_duplicates()
)

nb_accidents_multiples = (
    df_lieux_selection_sans_doublons["Num_Acc"]
    .value_counts()
    .gt(1)
    .sum()
)

print("Lignes après suppression simulée :", len(df_lieux_selection_sans_doublons))
print("Num_Acc encore multiples :", nb_accidents_multiples)

Lignes après suppression simulée : 294438
Num_Acc encore multiples : 25458


#### Résultat

La sélection des 9 variables retenues fait apparaître **6 234 lignes redondantes**, qui pourront être supprimées lors du nettoyage.

Après cette suppression simulée, le jeu de données contiendrait **294 438 lignes**. Cependant, **25 458 accidents présentent encore plusieurs enregistrements**, car leurs lieux diffèrent sur au moins une des caractéristiques retenues.

La sélection des variables réduit donc les redondances sans permettre d'obtenir systématiquement une ligne unique par accident. Les enregistrements présentant des caractéristiques différentes seront conservés afin de ne pas perdre d'information pertinente.

### 2.4. Contrôle des valeurs manquantes

Les valeurs manquantes sont analysées uniquement sur les variables retenues pour la suite du projet.

Cette approche permet de concentrer les contrôles qualité sur les données réellement utiles à l'analyse.

In [1267]:
# 2.4. Contrôle des valeurs manquantes

df_lieux_selection.isna().sum()

Num_Acc    0
catr       0
circ       0
nbv        0
prof       0
plan       0
surf       0
situ       0
vma        0
dtype: int64

#### Résultat

Aucune valeur manquante détectée par pandas n'est présente dans les **9 variables retenues**.

Cependant, l'absence de `NaN` ne signifie pas que toutes les informations sont renseignées. Certaines variables BAAC utilisent des codes spécifiques, notamment `-1`, pour représenter des valeurs non renseignées.

Ces codes seront analysés séparément.

### 2.5. Contrôle des valeurs codées `-1`

Certaines variables BAAC utilisent la valeur `-1` pour représenter une information non renseignée.

La présence de ce code est vérifiée uniquement sur les variables retenues avant toute décision de nettoyage.

In [1268]:
# 2.5. Contrôle des valeurs codées -1

for colonne in colonnes_lieux_retenues:
    nombre = (
        df_lieux_selection[colonne]
        .astype("string")
        .str.strip()
        .eq("-1")
        .sum()
    )

    if nombre > 0:
        print(f"{colonne} : {nombre}")

circ : 18405
nbv : 9911
prof : 285
plan : 227
surf : 289
situ : 262
vma : 10720


#### Résultat

Le code `-1`, correspondant à une information non renseignée, est présent dans **7 des variables retenues** : `circ`, `nbv`, `prof`, `plan`, `surf`, `situ` et `vma`.

Ces valeurs seront traitées comme des valeurs manquantes lors du nettoyage.

Les autres codes, notamment `0`, ne seront pas considérés automatiquement comme des valeurs manquantes : leur signification dépend de chaque variable.

### 2.6. Contrôle des modalités des variables catégorielles

Les modalités des variables catégorielles retenues sont examinées afin de vérifier la cohérence des codes présents dans les données avec la documentation BAAC.

Ce contrôle permet notamment d'identifier d'éventuelles valeurs inattendues avant le nettoyage.

In [1269]:
# 2.6. Contrôle des modalités des variables catégorielles

colonnes_categorielles = [
    "catr",
    "circ",
    "prof",
    "plan",
    "surf",
    "situ"
]

for colonne in colonnes_categorielles:
    print(f"{colonne} :")
    print(df_lieux_selection[colonne].value_counts().sort_index())
    print()

catr :
catr
1     25288
2     18148
3    113353
4    130806
5       322
6      1975
7      9376
9      1404
Name: count, dtype: int64

circ :
circ
-1     18405
 1     56006
 2    186302
 3     38135
 4      1824
Name: count, dtype: int64

prof :
prof
-1       285
 1    244808
 2     46785
 3      4710
 4      4084
Name: count, dtype: int64

plan :
plan
-1       227
 1    244112
 2     27554
 3     25031
 4      3748
Name: count, dtype: int64

surf :
surf
-1       289
 1    240150
 2     55987
 3       474
 4       143
 5       427
 6       245
 7      1024
 8       420
 9      1513
Name: count, dtype: int64

situ :
situ
-1       262
 1    249657
 2      1692
 3     19803
 4      6594
 5      7715
 6      3867
 8     11082
Name: count, dtype: int64



#### Résultat

Les modalités observées pour les variables catégorielles `catr`, `circ`, `prof`, `plan`, `surf` et `situ` sont cohérentes avec les codes prévus dans la documentation BAAC.

Aucun code inattendu n'a été identifié. Le code `-1`, présent dans certaines variables, correspond à une information non renseignée et sera traité lors du nettoyage.

L'absence de certaines modalités prévues dans la documentation ne constitue pas une anomalie : elles ne sont simplement pas représentées dans les données étudiées.

### 2.7. Contrôle des variables quantitatives

Les variables quantitatives retenues sont contrôlées séparément afin de vérifier leur format et d'identifier les éventuelles valeurs non exploitables.

#### 2.7.1. Nombre de voies de circulation (`nbv`)

La variable `nbv` représente le nombre de voies de circulation. Sa conversion en valeur numérique est testée afin d'identifier les éventuelles valeurs ne correspondant pas à un nombre exploitable.

In [1270]:
# 2.7.1. Contrôle de la variable nbv

nbv_numerique = pd.to_numeric(
    df_lieux_selection["nbv"],
    errors="coerce"
)

df_lieux_selection.loc[
    nbv_numerique.isna(),
    "nbv"
].value_counts()

nbv
#VALEURMULTI    104
#ERREUR           1
Name: count, dtype: int64

##### Résultat

La variable `nbv` contient **105 valeurs non numériques** : 104 occurrences de `#VALEURMULTI` et 1 occurrence de `#ERREUR`.

À ces valeurs s'ajoutent les codes `-1`, correspondant à une information non renseignée.

Les valeurs `#VALEURMULTI`, `#ERREUR` et `-1` seront converties en valeurs manquantes lors du nettoyage. Les valeurs numériques valides seront conservées.

#### 2.7.2. Vitesse maximale autorisée (`vma`)

La variable `vma` représente la vitesse maximale autorisée sur le lieu de l'accident.

Sa distribution est examinée afin d'identifier les valeurs non renseignées et d'éventuelles vitesses atypiques ou incohérentes avant le nettoyage.

In [1271]:
# 2.7.2. Contrôle de la variable vma

df_lieux_selection["vma"].value_counts().sort_index()

vma
-1       10720
 0           1
 1          72
 2          39
 3          14
 4           3
 5         117
 6          28
 7           2
 8           2
 9           1
 10        540
 12          1
 15        162
 16          1
 20        981
 23          1
 25        211
 30      45788
 31          1
 35         15
 40        306
 45         90
 50     145700
 55          3
 60        661
 65          2
 70      19852
 75          5
 80      38944
 85          1
 90      21896
 95          1
 100        46
 110      9343
 130      5046
 140         2
 180         1
 300         9
 301         1
 500        47
 501         1
 502         1
 520         1
 700         4
 770         1
 800         1
 900         6
 901         1
Name: count, dtype: int64

In [1272]:
# Vérification des valeurs atypiques de vma

df_lieux_selection.loc[
    df_lieux_selection["vma"] > 130,
    "vma"
].value_counts().sort_index()

vma
140     2
180     1
300     9
301     1
500    47
501     1
502     1
520     1
700     4
770     1
800     1
900     6
901     1
Name: count, dtype: int64

In [1273]:
# Vérification des valeurs atypiques de vma selon l'année

vma_atypiques = df_lieux_selection.loc[
    df_lieux_selection["vma"] > 130,
    ["Num_Acc", "vma"]
].copy()

vma_atypiques["annee"] = (
    vma_atypiques["Num_Acc"]
    .astype("string")
    .str[:4]
)

pd.crosstab(
    vma_atypiques["annee"],
    vma_atypiques["vma"]
)

vma,140,180,300,301,500,501,502,520,700,770,800,900,901
annee,,,,,,,,,,,,,
2020,0,0,1,0,11,0,0,0,0,0,0,0,0
2021,1,1,1,0,7,1,1,1,2,1,0,4,1
2022,0,0,5,0,8,0,0,0,1,0,0,0,0
2024,1,0,2,1,21,0,0,0,1,0,1,2,0


##### Résultat

La variable `vma` contient **10 720 valeurs `-1`**, correspondant à une information non renseignée.

L'analyse de sa distribution a également permis d'identifier **76 valeurs supérieures à 130 km/h**, considérées comme aberrantes au regard de la signification métier de la variable. Elles représentent environ **0,03 % des observations**.

Certaines de ces valeurs pourraient provenir d'erreurs de saisie, mais leur valeur correcte ne peut pas être déterminée avec certitude.

#### Décision

Lors du nettoyage :
- les valeurs `-1` seront converties en valeurs manquantes ;
- les 76 valeurs supérieures à 130 km/h seront également converties en valeurs manquantes ;
- aucune correction arbitraire des valeurs aberrantes ne sera effectuée ;
- la variable `vma` sera conservée pour la suite de l'analyse.

### 2.8. Synthèse des décisions de nettoyage

À la suite des contrôles qualité, les décisions suivantes sont retenues :

| Élément | Constat | Décision |
|---|---|---|
| Variables | 9 variables retenues pour l'analyse | Conserver uniquement les variables sélectionnées |
| Doublons | 6 234 lignes deviennent redondantes après sélection | Supprimer les doublons complets après sélection |
| Valeurs `-1` | Informations non renseignées dans plusieurs variables | Convertir en valeurs manquantes |
| `nbv` | `#VALEURMULTI`, `#ERREUR` et `-1` sont non exploitables | Convertir en valeurs manquantes, sans imputation |
| `vma` | `-1` correspond à une information non renseignée | Convertir en valeur manquante |
| `vma > 130` | 76 valeurs aberrantes, soit environ 0,03 % | Convertir en valeurs manquantes sans correction arbitraire |
| Types de données | Certains types sont hétérogènes ou non adaptés à l'analyse | Harmoniser les types lors du nettoyage |
| Granularité | Certains accidents possèdent plusieurs lignes `Lieux` avec des caractéristiques différentes | Conserver ces lignes dans la table nettoyée et traiter l'unicité de `Num_Acc` ultérieurement dans la construction de la table analytique |

La suppression individuelle des variables retenues ne permet pas d'obtenir une ligne unique par accident. La multiplicité des `Num_Acc` résulte donc de différences portant sur plusieurs caractéristiques des lieux.

La table `Lieux` nettoyée conservera sa granularité afin de ne pas supprimer arbitrairement des informations. Une table analytique à la granularité de l'accident sera construite ultérieurement pour permettre les jointures et analyses nécessitant un `Num_Acc` unique.

## 3. Nettoyage des données

Cette étape applique les décisions prises lors de l'évaluation de la qualité des données.

Les traitements concernent la sélection des variables, la suppression des lignes redondantes, le traitement des valeurs non renseignées ou aberrantes et l'harmonisation des types de données.

In [1274]:
# 3.1. Sélection des variables retenues

df_lieux_clean = df_lieux[
    colonnes_lieux_retenues
].copy()

df_lieux_clean.shape

(300672, 9)

### 3.2. Suppression des lignes redondantes

Après la sélection des variables, certaines lignes deviennent strictement identiques.

Ces lignes redondantes sont supprimées afin d'éviter de compter plusieurs fois une même combinaison de caractéristiques pour un accident.

In [1275]:
# 3.2. Suppression des lignes redondantes

df_lieux_clean = (
    df_lieux_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

df_lieux_clean.shape

(294438, 9)

### 3.3. Traitement des valeurs non renseignées

Dans les variables BAAC retenues, le code `-1` correspond à une information non renseignée.

Ces valeurs sont remplacées par des valeurs manquantes afin de les distinguer des modalités réellement observées.

In [1276]:
# 3.3.1. Traitement des valeurs codées -1

colonnes_avec_moins_un = [
    "circ",
    "nbv",
    "prof",
    "plan",
    "surf",
    "situ",
    "vma"
]

for colonne in colonnes_avec_moins_un:
    df_lieux_clean[colonne] = (
        df_lieux_clean[colonne]
        .astype("string")
        .str.strip()
        .replace("-1", pd.NA)
    )

In [1277]:
# Vérification du traitement des valeurs codées -1

for colonne in colonnes_avec_moins_un:
    nombre = (
        df_lieux_clean[colonne]
        .astype("string")
        .str.strip()
        .eq("-1")
        .sum()
    )

    print(f"{colonne} : {nombre}")

circ : 0
nbv : 0
prof : 0
plan : 0
surf : 0
situ : 0
vma : 0


In [1278]:
# 3.3.2 Traitement des valeurs non exploitables de nbv

df_lieux_clean["nbv"] = (
    df_lieux_clean["nbv"]
    .replace(["#VALEURMULTI", "#ERREUR"], pd.NA)
)

In [1279]:
# Vérification du traitement des valeurs non exploitables de nbv

df_lieux_clean["nbv"].isin(
    ["#VALEURMULTI", "#ERREUR"]
).sum()

np.int64(0)

In [1280]:
# 3.3.3. Traitement des valeurs aberrantes de vma

df_lieux_clean["vma"] = pd.to_numeric(
    df_lieux_clean["vma"],
    errors="coerce"
)

df_lieux_clean.loc[
    df_lieux_clean["vma"] > 130,
    "vma"
] = pd.NA

In [1281]:
# Vérification du traitement des valeurs aberrantes de vma

(df_lieux_clean["vma"] > 130).sum()

np.int64(0)

### 3.4. Harmonisation des types de données

In [1282]:
# Vérification des types avant harmonisation

df_lieux_clean.dtypes

Num_Acc     int64
catr        int64
circ       string
nbv        string
prof       string
plan       string
surf       string
situ       string
vma         Int64
dtype: object

In [1283]:
# 3.4. Harmonisation des types de données

df_lieux_clean["Num_Acc"] = (
    df_lieux_clean["Num_Acc"]
    .astype("string")
)

colonnes_entieres = [
    "catr",
    "circ",
    "nbv",
    "prof",
    "plan",
    "surf",
    "situ",
    "vma"
]

for colonne in colonnes_entieres:
    df_lieux_clean[colonne] = pd.to_numeric(
        df_lieux_clean[colonne],
        errors="coerce"
    ).astype("Int64")

In [1284]:
# Vérification des types après harmonisation

df_lieux_clean.dtypes

Num_Acc    string
catr        Int64
circ        Int64
nbv         Int64
prof        Int64
plan        Int64
surf        Int64
situ        Int64
vma         Int64
dtype: object

### 3.5. Renommage des variables

Les variables descriptives sont renommées avec des intitulés explicites afin de faciliter leur compréhension et leur utilisation dans les prochaines étapes du projet.

L'identifiant `Num_Acc` conserve son nom d'origine afin de maintenir une clé commune entre les différentes tables BAAC.

In [1285]:
# 3.5. Renommage des variables

df_lieux_clean = df_lieux_clean.rename(
    columns={
        "catr": "categorie_route",
        "circ": "regime_circulation",
        "nbv": "nombre_voies",
        "prof": "profil_route",
        "plan": "trace_route",
        "surf": "etat_surface",
        "situ": "situation_accident",
        "vma": "vitesse_max_autorisee"
    }
)

df_lieux_clean.columns.tolist()

['Num_Acc',
 'categorie_route',
 'regime_circulation',
 'nombre_voies',
 'profil_route',
 'trace_route',
 'etat_surface',
 'situation_accident',
 'vitesse_max_autorisee']

### 3.6. Validation finale du nettoyage

Une vérification finale est réalisée afin de s'assurer que les traitements appliqués ont produit un jeu de données cohérent avant son export.

In [1286]:
# 3.6.1. Vérification de la structure finale

print("Dimensions :", df_lieux_clean.shape)
print("Nombre de colonnes :", df_lieux_clean.shape[1])
print("Nombre de Num_Acc uniques :", df_lieux_clean["Num_Acc"].nunique())

Dimensions : (294438, 9)
Nombre de colonnes : 9
Nombre de Num_Acc uniques : 268788


In [1287]:
# 3.6.2. Vérification des valeurs manquantes

df_lieux_clean.isna().sum()

Num_Acc                      0
categorie_route              0
regime_circulation       18166
nombre_voies              9950
profil_route               285
trace_route                227
etat_surface               289
situation_accident         262
vitesse_max_autorisee    10761
dtype: int64

In [1288]:
# 3.6.3. Vérification des doublons complets

df_lieux_clean.duplicated().sum()

np.int64(0)

In [1289]:
# 3.6.4. Vérification des règles de nettoyage

print(
    "Valeurs vitesse_max_autorisee > 130 :",
    (df_lieux_clean["vitesse_max_autorisee"] > 130).sum()
)

print(
    "Num_Acc manquants :",
    df_lieux_clean["Num_Acc"].isna().sum()
)

Valeurs vitesse_max_autorisee > 130 : 0
Num_Acc manquants : 0


#### Bilan du nettoyage

Après nettoyage, la table `Lieux` contient **294 438 lignes et 9 variables**, correspondant à **268 788 accidents distincts**.

Les lignes devenues redondantes après la sélection des variables ont été supprimées. Les valeurs non renseignées ou non exploitables ont été converties en valeurs manquantes et les valeurs incohérentes de `vitesse_max_autorisee` supérieures à 130 km/h ont également été traitées.

Les types de données ont été harmonisés et aucun doublon complet ne subsiste.

Plusieurs lignes peuvent néanmoins être associées à un même `Num_Acc` lorsque les caractéristiques des lieux diffèrent. Cette granularité est volontairement conservée afin de ne pas perdre d'information. L'obtention d'une ligne unique par accident sera traitée ultérieurement lors de la construction de la table analytique.

In [1291]:
# 3.7. Export des données nettoyées

df_lieux_clean.to_csv(
    "data/processed/lieux_2020_2024_clean.csv",
    sep=";",
    index=False
)
df_lieux_clean.head() 

,Num_Acc,categorie_route,regime_circulation,nombre_voies,profil_route,trace_route,etat_surface,situation_accident,vitesse_max_autorisee
0,202000000001,4,2,2,1,1,1,1,50
1,202000000002,4,2,2,1,3,1,1,50
2,202000000003,4,<NA>,2,1,1,1,1,50
3,202000000004,4,2,2,1,1,1,1,30
4,202000000005,3,1,1,1,2,1,8,50
